# <center> Bit Level Manipulation
## <center> SYSE 549: Secure Vehicle and Industrial Networking
## <center> <img src="https://www.engr.colostate.edu/~jdaily/Systems-EN-CSU-1-C357.svg" width="400" />
### <center> Instructor: Dr. Jeremy Daily

## Learning Objectives

By the end of this lesson, students should be able to:

1. Understand bitwise boolean operations.
2. Apply masks and shifts to determine switch states.
3. Write code to set bits, clear bits, and toggle bits.
4. Decompose a 29-bit CAN identifier into its J1939 fields.
5. Read the position notation used in the J1939 standard and translate it into Python indices.

### Where the previous notebook left off

Notebook 01 treated the smallest unit of meaning as one byte. That is not fine enough for vehicle
networks. A CAN frame carries 8 bytes, and the J1939 standard routinely packs *four independent
parameters into a single byte* — two bits each.

The identifier is worse. It is 29 bits, and its internal fields do not land on byte boundaries.
You cannot slice it with `struct` alone.

So the tool for this notebook is the mask-and-shift: isolate the bits you want, move them down to
the bottom, and read the result. Every protocol analyzer you will ever use does exactly this,
several thousand times a second.

---
## 1. A Sample Message

Everything below works from one real frame captured on a truck.

The line format is `(timestamp) channel identifier [length] data-bytes`. As in Notebook 01, we
split on whitespace and rebuild the data field from the trailing hex text.

In [1]:
# A J1939 message from a CAN log.
msg = '(012.565071)  can1  18FEF100   [8]  FF 00 00 FC FF 68 00 CF'

In [2]:
# split() with no argument splits on runs of whitespace, so the double
# spaces do not produce empty fields.
split_message = msg.split()
split_message

['(012.565071)',
 'can1',
 '18FEF100',
 '[8]',
 'FF',
 '00',
 '00',
 'FC',
 'FF',
 '68',
 '00',
 'CF']

In [3]:
# Rebuild the 8-byte data field from the last 8 fields of the split line.
# Prefer this to a fixed slice such as msg[-24:], which silently breaks
# the moment the log format changes.
msg_bytes = bytes.fromhex(''.join(split_message[-8:]))
msg_bytes

b'\xff\x00\x00\xfc\xffh\x00\xcf'

### 1.1 `bytes` versus `bytearray`

A `bytes` object is **immutable**. Once created it cannot be changed, which is exactly what you
want for captured evidence: nothing you do downstream can quietly alter the record of what was on
the wire.

When you genuinely need to build or modify a message — constructing a frame to transmit, for
instance — use a `bytearray`.

In [4]:
# Bytes are immutable, so assignment fails.
# EXPECT AN EXCEPTION HERE. Read the message; it names the type as the problem.
msg_bytes[1] = 35

TypeError: 'bytes' object does not support item assignment

In [5]:
# A bytearray is the mutable counterpart. Same content, different guarantees.
msg_byte_array = bytearray.fromhex(''.join(split_message[-8:]))
msg_byte_array

bytearray(b'\xff\x00\x00\xfc\xffh\x00\xcf')

In [9]:
# Byte arrays support item assignment.
msg_byte_array[1] = 0x8D
msg_byte_array

bytearray(b'\xff\x8d\x00\xfc\xffh\x00\xcf')

We do not want to change the captured message, so from here on we work with `msg_bytes`, the
immutable copy.

Keeping evidence immutable is not a Python nicety. If you ever have to explain in a deposition
how a decoded value was produced, "the captured bytes could not be modified by the analysis code"
is a sentence worth being able to say.

---
## 2. How J1939 Numbers Things

Before touching the identifier, get the counting conventions straight. Nearly every bit-level bug
in student code traces back to one of these.

### Bytes

| J1939 | Python |
| :--- | :--- |
| Byte 1 | `msg_bytes[0]` |
| Byte 2 | `msg_bytes[1]` |
| Byte 4 | `msg_bytes[3]` |

**J1939 counts bytes from 1. Python indexes from 0.** Subtract one.

### Bits within a byte

J1939 numbers bits 1 through 8, with **bit 1 as the least significant bit**:

```
  bit number:    8   7   6   5   4   3   2   1
  place value: 128  64  32  16   8   4   2   1
  Python shift:  7   6   5   4   3   2   1   0
```

**Shift amount = bit number − 1.**

### Position notation

The standard writes a parameter's location as `byte.bit`. So SPN 597 at position **4.5** starts at
byte 4, bit 5. Since it is a 2-bit parameter, it occupies bits 5 and 6 of byte 4 — which is
`msg_bytes[3]`, shifted right by 4.

### Two-bit parameter states

Most switches and status flags in J1939 are 2 bits wide, which gives four states:

| Bits | Meaning |
| :--- | :--- |
| `00` | Off / disabled / no |
| `01` | On / enabled / yes |
| `10` | Error |
| `11` | **Not available** — the transmitter does not have this information |

`11` is not a failure and it is not "off." It means the sending ECU is not the authority for that
parameter. Treating "not available" as "off" is a genuine source of wrong conclusions in crash
reconstruction, and you will see it happen in Section 5.

---
## 3. The 29-Bit CAN Identifier

Please download SAE J1939-21 from the CSU Library:

https://saemobilus-sae-org.ezproxy2.library.colostate.edu/content/J1939/21_202205

An extended CAN identifier is 29 bits, written as 8 hex digits (32 bits) with the top 3 bits
unused. J1939 divides it like this:

| Bits (0 = LSB) | Width | Field |
| :--- | :--- | :--- |
| 26–28 | 3 | Priority (0 = highest, 7 = lowest) |
| 25 | 1 | EDP — Extended Data Page |
| 24 | 1 | DP — Data Page |
| 16–23 | 8 | PF — PDU Format |
| 8–15 | 8 | PS — PDU Specific |
| 0–7 | 8 | SA — Source Address |

Notice that priority, EDP, and DP all live inside the **first byte** and none of them is byte
aligned. That is the reason this notebook exists.

We will extract these two ways:

1. Convert to raw bytes and use `struct.unpack` — fast, but it cannot see inside a byte.
2. Convert to an integer and use masks and shifts — works at any bit boundary.

In [10]:
# struct is used for the first approach.
import struct

In [11]:
# Approach 1: treat the identifier as 4 raw bytes.
can_id = bytes.fromhex(split_message[2])
can_id

b'\x18\xfe\xf1\x00'

In [12]:
# PDU = Protocol Data Unit.
# Unpack the four bytes. Note PR is the whole first byte, not just the priority.
(PR, PF, PS, SA) = struct.unpack("BBBB", can_id)
print("Source Address = {}".format(SA))
print("PDU Specific   = {}".format(PS))
print("PDU Format     = {}".format(PF))
print("First byte     = {}".format(PR))

Source Address = 0
PDU Specific   = 241
PDU Format     = 254
First byte     = 24


In [13]:
# Show the first byte in binary. Only some of these bits are the priority.
print("{:08b}".format(PR))

00011000


Priority is only 3 bits (values 0–7), so the first byte holds more than we asked for. Reading
`PR` as the priority would report 24 for a message whose priority is 6.

Mask off the three bits we want and shift them down. The mask `0b00011100` has ones exactly where
the priority lives; the `>> 2` moves bit 2 to bit 0 so the result reads as a plain number.

In [14]:
# Mask the 3 priority bits, then shift them down to the bottom.
priority = (PR & 0b00011100) >> 2
priority

6

That worked, but notice what we had to do: unpack to bytes, then *still* mask and shift inside a
byte. The two leftover bits of that first byte are EDP and DP, and they are not decoration — they
are part of the PGN.

If we are going to mask and shift anyway, it is cleaner to skip `struct` entirely and treat the
whole identifier as one 29-bit integer.

In [15]:
# Approach 2: convert the identifier text directly to an integer, base 16.
can_id = int(split_message[2], 16)
can_id

419361024

In [16]:
# Confirm we still have the value we started with.
print("{:08X}".format(can_id))

18FEF100


In [17]:
# Source address is the lowest byte: no shift needed.
SA = (can_id & 0x000000FF)
SA

0

In [18]:
# Mask the second byte for PDU Specific.
# Masking alone leaves the value sitting in the wrong place.
PS = (can_id & 0xFF00)
PS

61696

In [19]:
# Shown as hex, the problem is obvious: the value is 0xF1 followed by two zeros.
print("{:X}".format(PS))

F100


In [20]:
# Shift the masked result down by one byte (8 bits).
PS = (can_id & 0xFF00) >> 8
PS

241

In [21]:
# Now it reads as a plain number.
print("{:X}".format(PS))

F1


### 3.1 Bringing in the Data Page bits

For the PDU Format we deliberately mask **10 bits**, not 8: the 8 bits of PF plus the DP and EDP
bits sitting just above them. Those two bits select which page of the standard the PGN comes from,
and leaving them out means you cannot distinguish PGNs on page 0 from page 1.

Mask `0x03FF0000` covers bits 16 through 25.

In [22]:
# Include the DP and EDP bits above PF: 10 bits total, bits 16-25.
mask = 0x3FF0000
print("Mask: {:032b}".format(mask))
print("Data: {:032b}".format(can_id))

Mask: 00000011111111110000000000000000
Data: 00011000111111101111000100000000


Perform a bitwise AND.

Everywhere there is a zero in the mask, the result is zeroed.

Everywhere there is a one in the mask, the result keeps the data bit.

Line the three rows up and read down the columns. This is the whole idea.

In [23]:
# Apply the mask and print all three rows aligned so the columns line up.
PF = mask & can_id
print("Mask: {:032b}".format(mask))
print("Data: {:032b}".format(can_id))
print("Rslt: {:032b}".format(PF))
PF

Mask: 00000011111111110000000000000000
Data: 00011000111111101111000100000000
Rslt: 00000000111111100000000000000000


16646144

In [24]:
# Shift the masked result down by 16 bits.
PF = (can_id & mask) >> 16
PF

254

In [25]:
# 254 == 0xFE. For this message DP and EDP are both 0, so the 10-bit
# field and the 8-bit PF happen to agree.
print("{:X}".format(PF))

FE


In [26]:
# Priority again, this time straight from the integer. No struct required.
priority_mask = 0x1C000000
PR = priority_mask & can_id
print("Mask: {:032b}".format(priority_mask))
print("Data: {:032b}".format(can_id))
print("Rslt: {:032b}".format(PR))

Mask: 00011100000000000000000000000000
Data: 00011000111111101111000100000000
Rslt: 00011000000000000000000000000000


In [27]:
# Shift down 26 bits (8 + 8 + 8 + 2) to read the priority as 0-7.
priority = PR >> 26
priority

6

---
## 4. Assembling the PGN

The Parameter Group Number identifies *what the message is about*. How it is built from PF and PS
depends on the value of PF:

| Condition | Called | Rule |
| :--- | :--- | :--- |
| PF ≥ 240 (0xF0) | **PDU2**, broadcast | PGN = PF and PS concatenated |
| PF < 240 | **PDU1**, destination specific | PS is a *destination address*, not part of the PGN. PGN = PF with the low byte set to 0 |

Our PF is 254, so this is PDU2 and the PGN is PF and PS joined together.

Getting the PDU1 case wrong is a classic error: a request message with identifier `18EAFF00` has
PF = 0xEA = 234, so its PGN is 0xEA00 = 59904, **not** 0xEAFF. The 0xFF is the global destination
address.

In [28]:
# A first attempt at concatenating PF and PS. Predict the answer before running.
PGN = PF << 8 + PS
PGN

229774927080299325293648673376615067146332625820880494265798643265702616629248

That is not a 16-bit number.

Operator precedence bit us: `+` binds more tightly than `<<`, so Python evaluated `8 + PS` first
and computed `254 << 249` — a 257-bit integer. Python's arbitrary-precision integers were happy
to oblige, and produced a result with no warning at all.

Reference: https://en.cppreference.com/w/c/language/operator_precedence

Addition takes place before bit shifting. Bit shifting takes place before AND. When in doubt,
parenthesize; nobody has ever been criticized for excessive clarity in a mask expression.

In [29]:
# Order of operations matters. Parentheses force the shift to happen first.
PGN = (PF << 8) + PS #Combine the PDU Format and PDU Specific Section
PGN += DP << 9 # Add the Datapage value
PGN += EDP << 10 #Add the extended data page value

65265

In [ ]:
# 65265 == 0xFEF1. Look this up in J1939-71.
print("PGN {} = 0x{:04X}".format(PGN, PGN))

---
## 5. Decoding the Data Field

The J1939 standard has this to say about PGN 65265 (0xFEF1):

**Name:** Cruise Control / Vehicle Speed 1 &nbsp;&nbsp;&nbsp; **Acronym:** CCVS1

It contains these suspect parameter numbers (SPNs):

| Position | SPN | SP Label | SP Length | Resolution |
| :--- | :--- | :--- | :--- | :--- |
| 1.1 | 69 | Two Speed Axle Switch | 2 bits | 4 states/2 bit |
| 1.3 | 70 | Parking Brake Switch | 2 bits | 4 states/2 bit |
| 1.5 | 1633 | Cruise Control Pause Switch | 2 bits | 4 states/2 bit |
| 1.7 | 3807 | Park Brake Release Inhibit Request | 2 bits | 4 states/2 bit |
| 2–3 | 84 | Wheel-Based Vehicle Speed | 2 bytes | 1/256 km/h per bit |
| 4.1 | 595 | Cruise Control Active | 2 bits | 4 states/2 bit |
| 4.3 | 596 | Cruise Control Enable Switch | 2 bits | 4 states/2 bit |
| 4.5 | 597 | Brake Switch | 2 bits | 4 states/2 bit |
| 4.7 | 598 | Clutch Switch | 2 bits | 4 states/2 bit |

Byte 4 alone carries four independent parameters. This is why the mask-and-shift matters.

In [ ]:
# Recall the captured data field.
msg_bytes

In [ ]:
# SPN 597, Brake Switch, is at position 4.5 -> byte 4, bit 5.
# J1939 byte 4 is Python index 3.
byte_of_interest = msg_bytes[3]
print("{:08b}".format(byte_of_interest))

In [ ]:
# Bit 5 in J1939 numbering is shift 4 in Python.
# A 2-bit parameter starting there needs a 2-bit mask: 0b11 << 4.
mask = 0b00110000
print("{:08b}".format(mask))

In [ ]:
# Apply the mask. Print all three rows aligned and read down the columns.
result = byte_of_interest & mask
print("{:08b}  mask".format(mask))
print("{:08b}  data".format(byte_of_interest))
print("{:08b}  result".format(result))

In [ ]:
# Shift the isolated bits down to positions 0 and 1 so they read as 0-3.
SPN597_value = result >> 4
SPN597_value

In [ ]:
# Per J1939, decode the 2-bit state into text.
SPN597_meanings = {0: "Brake pedal released",
                   1: "Brake pedal depressed",
                   2: "Error",
                   3: "Not Available"}
print(SPN597_meanings[SPN597_value])

### 5.1 Read that result carefully

The brake switch is **Not Available** — not "released."

The source address on this frame is 0x00, the engine controller. The engine is not the authority
for the brake pedal; that state is sourced from the cab or the brake system controller. So the
engine transmits `11` to say *I am not the one to ask.*

Now imagine an analyst who masks the wrong two bits, reads `00`, and writes "the brake was not
applied" into a report. The bits were right there in the capture. The error was in the decoding,
and nothing downstream would flag it.

This is the practical reason to line up the mask, the data, and the result and read down the
columns before trusting a number.

In [ ]:
# Decode all four 2-bit parameters in byte 4 with one loop.
# The shift is (bit position - 1); 0b11 is the 2-bit mask.
byte4_spns = [(595, "Cruise Control Active", 1),
              (596, "Cruise Control Enable", 3),
              (597, "Brake Switch",          5),
              (598, "Clutch Switch",         7)]
for spn, label, bit_position in byte4_spns:
    value = (byte_of_interest >> (bit_position - 1)) & 0b11
    print("SPN {:4d}  {:24s} = {} ({:02b})".format(spn, label, value, value))

### 5.2 A multi-byte parameter: SPN 84

SPN 84, *Wheel-Based Vehicle Speed*, spans bytes 2 and 3. It is a 2-byte little-endian unsigned
integer scaled at 1/256 km/h per bit — exactly the kind of field Notebook 01 covered.

Note the mixed convention inside a single protocol: the identifier is read most significant bit
first, and the data field is little endian. That is normal, and it is a reliable source of bugs.

In [ ]:
# Bytes 2-3 in J1939 are Python indices 1 and 2.
speed_bytes = msg_bytes[1:3]
speed_bytes

In [ ]:
# Little endian, unsigned, 2 bytes. Then apply the 1/256 km/h resolution.
speed_raw = struct.unpack('<H', speed_bytes)[0]
speed_kph = speed_raw / 256
print("Raw: {}   Speed: {:.3f} km/h".format(speed_raw, speed_kph))

### 5.3 The "not available" pattern at full width

Zero km/h is a legitimate reading — the truck is stopped. But what if those two bytes had been
`FF FF`?

J1939 reserves the all-ones pattern to mean *not available* for a parameter of any width, the
same way `11` does for a 2-bit switch. SPN 84 has a defined range topping out near 251 km/h, so
0xFFFF cannot be a speed.

Run the next cell and look at the number it produces.

In [ ]:
# What a naive decoder reports when the field is the not-available code.
not_available_raw = struct.unpack('<H', b'\xff\xff')[0]
print("Decoded as a speed: {:.3f} km/h".format(not_available_raw / 256))
print("Which is           {:.1f} mph".format(not_available_raw / 256 / 1.609344))

About 256 km/h, or 159 mph, from a loaded tractor. Absurdity is a diagnostic — the same lesson as
the 10^161 odometer in Notebook 01. A speed field of 0xFFFF means the parameter is not available,
and a decoder that reports 256 km/h has told you it does not know the standard.

Byte 1 of our captured message is 0xFF, which is all four of its 2-bit switches reporting
`11`. The engine has nothing to say about the parking brake or the two-speed axle either.

In [ ]:
# Byte 1 is 0xFF: four 2-bit parameters, all reporting "not available".
print("{:08b}".format(msg_bytes[0]))

### 5.4 One helper instead of many magic numbers

The pattern `(data >> shift) & mask` is the same every time. The only things that change are
where the field starts and how wide it is.

`(1 << width) - 1` is a compact way to build a mask of `width` ones: `1 << 2` is `0b100`, and
subtracting 1 gives `0b011`.

In [ ]:
# Generalized field extraction using J1939 position notation.
def extract_spn(data_bytes, byte_position, bit_position, width):
    """Return the unsigned value of a field inside a single byte.

    byte_position and bit_position use J1939 numbering (both start at 1).
    width is in bits.
    """
    byte_value = data_bytes[byte_position - 1]      # J1939 counts from 1
    shift = bit_position - 1                        # so does bit numbering
    mask = (1 << width) - 1                         # width ones, e.g. 0b11
    return (byte_value >> shift) & mask

In [ ]:
# Verify the helper reproduces the brake switch result from section 5.
extract_spn(msg_bytes, 4, 5, 2)

In [ ]:
# And confirm it matches the value we computed by hand.
extract_spn(msg_bytes, 4, 5, 2) == SPN597_value

---
## 6. Bit Manipulation

Reading bits is half the job. Transmitting a message means *setting* them, and usually setting one
parameter without disturbing the seven others sharing the byte.

| Goal | Operator | Idiom |
| :--- | :--- | :--- |
| Set a bit to 1 | OR | `x \|= (1 << n)` |
| Clear a bit to 0 | AND with NOT | `x &= ~(1 << n)` |
| Toggle a bit | XOR | `x ^= (1 << n)` |
| Test a bit | AND | `if x & (1 << n):` |

Reference: https://stackoverflow.com/questions/47981/how-do-you-set-clear-and-toggle-a-single-bit

In [ ]:
# Four single-bit switches packed into one byte, two bit positions apart.
switch = 0
sw1 = 0
sw2 = 1
sw3 = 0
sw4 = 1
# Initial build: shift each switch into place and add.
switch = sw1 + (sw2 << 2) + (sw3 << 4) + (sw4 << 6)
print("{:08b}".format(switch))

In [ ]:
# Pack it as a single unsigned byte, ready to place in a message.
struct.pack("B", switch)

In [ ]:
# Set Switch 1 (OR). OR with a 1 forces that bit high; OR with 0 leaves a bit alone.
switch |= 1
print("{:08b}".format(switch))

In [ ]:
# Set Switch 3, which lives at bit index 4.
switch |= 1 << 4
print("{:08b}".format(switch))

In [ ]:
# Clear Switch 4 (AND with the inverted mask).
# ~(1 << 6) is all ones except at bit 6, so only that bit is forced low.
switch &= ~(1 << 6)
print("{:08b}".format(switch))

In [ ]:
# Clear Switch 2 the same way.
switch &= ~(1 << 2)
print("{:08b}".format(switch))

In [ ]:
# Toggle Switch 3 (XOR). XOR with 1 flips a bit; XOR with 0 leaves it alone.
switch ^= 1 << 4
print("{:08b}".format(switch))

In [ ]:
# Toggle it back. You may pre-calculate the mask: (1 << 4) == 16 == 0b00010000.
switch ^= 16
print("{:08b}".format(switch))

### 6.1 A Python-specific trap with `~`

In C, `~` on an 8-bit type produces an 8-bit result. Python integers have **arbitrary precision
and are signed**, so `~x` is `-x - 1` and the result is negative.

The clearing idiom above still works, because ANDing a value in 0–255 with a negative number never
escapes that range. But the moment you print or pack the inverted mask on its own, the difference
shows.

In [ ]:
# In Python, ~ produces a negative number rather than an 8-bit complement.
print(~1)
print(~(1 << 6))

In [ ]:
# Formatting a negative as binary gives a minus sign, not two's complement.
# This is NOT the 8-bit mask you may have expected.
print("{:b}".format(~(1 << 6)))

In [ ]:
# Mask with 0xFF to get the byte-sized inverted mask.
print("{:08b}".format(~(1 << 6) & 0xFF))

> **Rule of thumb:** when a result is going back into a byte, finish with `& 0xFF`. It costs
> nothing when the value is already in range, and it prevents `struct.error` when it is not.

### 6.2 Putting it together: build a CCVS1 byte 4

Everything in this notebook now converges. Construct byte 4 of a CCVS1 message that reports
cruise control active, cruise enabled, brake released, and clutch not available — then decode it
back with the helper from Section 5.4 to check the work.

In [ ]:
# Build byte 4 from four 2-bit parameters, using J1939 bit positions.
byte4 = 0
byte4 |= 1 << (1 - 1)   # SPN 595 Cruise Control Active = 01 (active)
byte4 |= 1 << (3 - 1)   # SPN 596 Cruise Control Enable = 01 (enabled)
byte4 |= 0 << (5 - 1)   # SPN 597 Brake Switch          = 00 (released)
byte4 |= 3 << (7 - 1)   # SPN 598 Clutch Switch         = 11 (not available)
print("byte 4 = 0x{:02X} = {:08b}".format(byte4, byte4))

In [ ]:
# Assemble a full 8-byte CCVS1 data field with a wheel speed of 65.0 km/h.
# 65.0 km/h / (1/256) = 16640, packed little endian as 00 41.
data = bytearray(b'\xFF' * 8)            # start with everything not available
data[1:3] = struct.pack('<H', int(65.0 * 256))   # SPN 84, bytes 2-3
data[3] = byte4                                  # byte 4
frame = bytes(data)                              # freeze it
frame.hex(' ').upper()

In [ ]:
# Decode the frame we just built and confirm it round trips.
print("Speed: {:.1f} km/h".format(struct.unpack('<H', frame[1:3])[0] / 256))
for spn, label, bit_position in byte4_spns:
    print("SPN {:4d}  {:24s} = {}".format(spn, label,
                                          extract_spn(frame, 4, bit_position, 2)))

---
## Summary

You can manipulate bits using OR, AND, and XOR. You can read bits using masks and shifts.

In this notebook we used masks and bit shifts to read and construct data encoded inside bytes,
working from a real J1939 message. Two ideas are worth carrying forward:

* **The meaning of the data comes from the standard, not from the bytes.** J1939-21 told us how
  to split the identifier; J1939-71 told us what byte 4 of PGN 65265 contains. Without those
  documents the frame is 8 unlabeled bytes.
* **Line up the mask, the data, and the result, and read down the columns.** Almost every
  bit-level bug becomes visible the moment you print the three rows together.

And keep the conventions straight: bytes count from 1, bits count from 1, bit 1 is the least
significant, and the shift amount is always one less than the bit number.

## Check Your Understanding

Work these in the blank cells below.

1. A frame has the identifier `0CF00400`. Find the priority, PF, PS, SA, and the PGN. Is this
   PDU1 or PDU2, and how did you decide?
2. A frame has the identifier `18EAFF00`. Work out its PGN. Explain in one sentence why the
   answer is not `0xEAFF`.
3. Byte 4 of a CCVS1 message is `0xD6`. Decode all four of its SPNs and state what the brake
   switch reports.
4. Write one line that clears SPN 597 (position 4.5, 2 bits wide) to `00` in a byte named
   `b4`, leaving the other six bits untouched.
5. A decoder reports a wheel-based vehicle speed of 255.996 km/h. What is almost certainly in
   bytes 2–3 of that frame, and what should the decoder have reported instead?